# Import the necessary libraries and modules

In [1]:
import torch  # PyTorch library for deep learning
import torch.nn as nn  # Neural network modules
import numpy as np  # NumPy library for numerical computations
import pennylane as qml  # PennyLane for quantum machine learning
import matplotlib.pyplot as plt  # Matplotlib for plotting
import pandas as pd  # Library for data manipulation and analysis  
dtype = torch.float  # Set default tensor data type to float

In [2]:
def Solution(x, y, params):
    # Analytical solution of the wave function for comparison
    Lx, Ly, w, l, m, h_bar, mass = params
    return (2/np.sqrt(Lx*Ly))*torch.sin((l*torch.pi/Lx)*x)*torch.sin((m*torch.pi/Ly)*y)

def E_func(params):
    # Analytical solution of energy for comparison
    Lx, Ly, w, l, m, h_bar, mass = params
    return (1/(2*mass))*((h_bar*torch.pi)**2)*((pow(l, 2)/Lx**2) + (pow(m, 2)/Ly**2))

def test_PINN(NN, domain, num_test, E_exact, Ep, psi_exact):
    # Evaluate the neural network against the analytical solution and compute error metrics
    x = torch.linspace(domain[0][0], domain[0][1], num_test)
    y = torch.linspace(domain[1][0], domain[1][1], num_test)
    
    X, Y = torch.meshgrid(x, y, indexing="ij")
    data = torch.vstack([X.ravel(), Y.ravel()]).T # reshape the grid into a 2D array of points
    with torch.no_grad():
        psi_pred, E_pred = NN(data)

    psi2_pred = psi_pred**2
    
    psi_pred = torch.reshape(psi_pred, (num_test, num_test))
    
    list_berror = []
    list_bmerror = []
    for psi_e in psi_exact:
        psi2_exact = (psi_e**2).view(-1, 1)
        error = torch.abs(psi2_exact - psi2_pred)
        mean_error = torch.mean(error)
        list_bmerror.append(mean_error)
    
    min_e = min(list_bmerror)
    min_idx = list_bmerror.index(min_e)
    psi2_merror = list_bmerror[min_idx]
        
    E_pred = E_pred.item() + Ep
    energy_error = abs((E_pred) - E_exact)
    
    return energy_error, psi2_merror

In [3]:
# Set the necessary parameters
energy_level = 3
num_test = 20
domain = [[0, 2],[0, 2]]

x = torch.linspace(domain[0][0], domain[0][1], num_test)
y = torch.linspace(domain[1][0], domain[1][1], num_test)

X, Y = torch.meshgrid(x, y, indexing="ij")
data = torch.vstack([X.ravel(), Y.ravel()]).T # reshape the grid into a 2D array of points

In [4]:
psi_exact_list = []
E_exact_list = []
degeneracy_psi = []

# Define constants and compute domain size
Lx, Ly, w, l, m, h_bar, mass = (domain[0][1] - domain[0][0]), (domain[1][1] - domain[1][0]), torch.pi, 1, 1, 1, 1
params = [Lx, Ly, w, l, m, h_bar, mass]

# Ground state: (l=1, m=1)
# Append exact wave function and energy
psi_exact_list.append([Solution(data[:, 0], data[:, 1], params)])
E_exact_list.append(E_func(params))

# First excited state (degenerate): (l=1, m=2) and (l=2, m=1)
# Both wave functions are added to a list and stored as a degenerate pair
Lx, Ly, w, l, m, h_bar, mass = (domain[0][1] - domain[0][0]), (domain[1][1] - domain[1][0]), torch.pi, 1, 2, 1, 1
params = [Lx, Ly, w, l, m, h_bar, mass]
degeneracy_psi.append(Solution(data[:, 0], data[:, 1], params))

Lx, Ly, w, l, m, h_bar, mass = (domain[0][1] - domain[0][0]), (domain[1][1] - domain[1][0]), torch.pi, 2, 1, 1, 1
params = [Lx, Ly, w, l, m, h_bar, mass]
degeneracy_psi.append(Solution(data[:, 0], data[:, 1], params))

# Append both degenerate wave functions and their common energy
psi_exact_list.append(degeneracy_psi)
E_exact_list.append(E_func(params))

# Second excited state: (l=2, m=2)
# Append exact wave function and energy
Lx, Ly, w, l, m, h_bar, mass = (domain[0][1] - domain[0][0]), (domain[1][1] - domain[1][0]), torch.pi, 2, 2, 1, 1
params = [Lx, Ly, w, l, m, h_bar, mass]
psi_exact_list.append([Solution(data[:, 0], data[:, 1], params)])
E_exact_list.append(E_func(params))

# Third excited state candidate: (l=3, m=1)
# Only energy is computed and stored here (wave function not added)
Lx, Ly, w, l, m, h_bar, mass = (domain[0][1] - domain[0][0]), (domain[1][1] - domain[1][0]), torch.pi, 3, 1, 1, 1
params = [Lx, Ly, w, l, m, h_bar, mass]
E4 = E_func(params)

# Classical Neural Network

In [5]:
class CNeuralNet(nn.Module):
    """
    Classical neural network class.

    Args:
        input_size (int): Size of the input features.
        hidden_size (int): Number of hidden units in the neural network.
        output_size (int): Size of the output.
        domain (list): Defined domain in the interval [0, 2] for x and y axis.

    Attributes:
        fnn (nn.Sequential): Feedforward neural network with two linear layer followed by a Tanh activation.

    Notes:
        - The classical neural network architecture consists of one linear layer for energy and two hidden layers for wave function.
        - The input layer has 'input_size' neurons.
        - The hidden layers for predicting wave function has 'hidden_size' neurons with a Tanh activation.
        - The output layer has 'output_size' neurons.
    """
    def __init__(self, input_size, hidden_size, output_size, domain, seed):
        super(CNeuralNet, self).__init__()

        self.seed = seed
        # Define the feedforward (1, 1) layer
        self.Energy = nn.Linear(1, 1)
        # Define the feedforward neural network (fnn)
        self.fnn = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, output_size))

        # Set the x-borders and y-borders based on the specified domain
        self.xborders = domain[0]
        self.yborders = domain[1]

    def forward(self, data):
        """
        Forward pass through the neural network.

        Args:
            data (torch.Tensor): Input data.

        Returns:
            torch.Tensor: Outputs predictions from the neural network.
        """
        # Extract input features
        x, y= data[:, 0].view(-1, 1), data[:, 1].view(-1, 1)
        
        # set lambda to pass through the linear layer (energy layer)
        lbd = torch.tensor([[1.0]])
        Energy = self.Energy(lbd)
        
        # Pass the data through the feedforward neural network (self.fnn) and combine outputs
        out = self.fnn(data)
        out1 = out[:, 0].view(-1, 1)
        out2 = out[:, 1].view(-1, 1)
        out = (1-torch.exp(-x+self.xborders[0]))*(1-torch.exp((x-self.xborders[1])))*out1*(1-torch.exp(-y+self.yborders[0]))*(1-torch.exp((y-self.yborders[1])))*out2

        return out, torch.abs(Energy)

    def init_weights(self, m):
        """
        Initializes weights and biases for a linear layer.

        Args:
            m (nn.Linear): The linear layer to initialize.

        Notes:
            - Applies uniform weight initialization within the range [-n0, n0].
            - Initializes biases with a constant value of 0.01.
        """
        if isinstance(m, nn.Linear):
            in_sz = torch.tensor(m.in_features)
            out_sz = torch.tensor(m.out_features)
            n0 = torch.sqrt(6/(in_sz+out_sz))
            m.weight.data.uniform_(-n0, n0)
            m.bias.data.fill_(0.01)

    def init_energy(self, shooting):
        """
        Initializes weights and biases for a linear layer.

        Args:
            m (nn.Linear): The linear layer to initialize.

        Notes:
            - Applies shooting number.
            - Initializes biases with a constant value of 0.0.
        """
        def inner(m):
            if isinstance(m, nn.Linear):
                m.weight.data.fill_(shooting)
                m.bias.data.fill_(0.0)
        return inner

    def set_net_params(self):
        """
        set weights and biases for linear layers.

        Args:
            None.
        """
        # Set the random seed for reproducibility
        torch.manual_seed(self.seed)
        self.fnn.apply(self.init_weights)

    def set_energy_params(self, shooting):
        """
        set shooting number for a linear layer.

        Args:
            shooting (float): The value of shooting energy.
        """
        self.Energy.apply(self.init_energy(shooting))

In [6]:
seeds = [14, 42, 86, 195]
classic_model_list_seeds = []
classic_history_lists_seeds = []
for seed in seeds:
    classic_model_list = []
    classic_history_lists = []
    for idx1, hs in enumerate([10, 30, 50]):
        idx1 = idx1 + 1
        input_size , hidden_size, output_size = 2, hs, 2
        energy_model_list = []
        for idx2 in range(energy_level):
            model_classic = CNeuralNet(input_size, hidden_size, output_size, domain, seed)
            model_classic.load_state_dict(torch.load(f"..\\trained_models\\classical time-tndependent schrödinger\\{seed}\\Classical model_{seed}_{hs}{idx2}.pth"))
            energy_model_list.append(model_classic)
        classic_model_list.append(energy_model_list)
        classic_history_lists.append(torch.load(f"..\\trained_models\\classical time-tndependent schrödinger\\{seed}\\loss_Classical model_{seed}_{hs}.pt"))
    
    classic_model_list_seeds.append(classic_model_list)
    classic_history_lists_seeds.append(classic_history_lists)

In [7]:
results = []
for idx1 in range(len(classic_model_list)):
    psi2_merror_list2 = []
    energy_error_list2 = []
    for idx2 in range(energy_level):
        psi2_merror_list3 = []
        energy_error_list3 = [] 
        for idx3, seed_name in enumerate(seeds):
            energy_error, psi2_merror = test_PINN(classic_model_list_seeds[idx3][idx1][idx2], domain, num_test, E_exact_list[idx2], classic_history_lists_seeds[idx3][idx1][idx2][-1], psi_exact_list[idx2])
            if energy_error >= 0.7:
                print("seed:",seed_name, "energy:", idx2+1, "Configuration:", idx1+1)
            else:
                psi2_merror_list3.append(psi2_merror)
                energy_error_list3.append(energy_error)
            
        mean_psi2 = np.mean(psi2_merror_list3)
        std_psi2 = np.std(psi2_merror_list3)

        mean_E = np.mean(energy_error_list3)
        std_E = np.std(energy_error_list3)
        psi2_merror_list2.append([idx2, mean_psi2, std_psi2])
        energy_error_list2.append([idx2, mean_E, std_E])

    results.append([idx1, psi2_merror_list2, energy_error_list2])

seed: 14 energy: 3 Configuration: 1
seed: 14 energy: 3 Configuration: 2
seed: 86 energy: 3 Configuration: 2


In [8]:
# Initialize lists to hold data for the two DataFrames
psi2_data = []
energy_data = []
hidden_sizes = [10, 30, 50]
# Loop over your results to extract and flatten the values
for idx1, psi2_list, energy_list in results:
    hidden_size = hidden_sizes[idx1]  # convert 0,1,2 → 10,30,50
    for state_idx, mean_psi2, std_psi2 in psi2_list:
        psi2_data.append([hidden_size, state_idx + 1, mean_psi2, std_psi2])
    for state_idx, mean_E, std_E in energy_list:
        energy_data.append([hidden_size, state_idx + 1, mean_E, std_E])

# Create DataFrames
df_psi2 = pd.DataFrame(psi2_data, columns=["Hidden Size", "State", "Mean ψ² Error", "Std. Dev. (ψ²)"])
df_energy = pd.DataFrame(energy_data, columns=["Hidden Size", "State", "Mean Energy Error", "Std. Dev. (Energy)"])

# Optional: Format with scientific notation
df_psi2["Mean ψ² Error"] = df_psi2["Mean ψ² Error"].map(lambda x: f"{x:.5e}")
df_psi2["Std. Dev. (ψ²)"] = df_psi2["Std. Dev. (ψ²)"].map(lambda x: f"{x:.5e}")
df_energy["Mean Energy Error"] = df_energy["Mean Energy Error"].map(lambda x: f"{x:.5e}")
df_energy["Std. Dev. (Energy)"] = df_energy["Std. Dev. (Energy)"].map(lambda x: f"{x:.5e}")

# Display the tables
print("ψ² Errors:")
print(df_psi2.to_string(index=False))

print("\nEnergy Errors:")
print(df_energy.to_string(index=False))


ψ² Errors:
 Hidden Size  State Mean ψ² Error Std. Dev. (ψ²)
          10      1   8.47240e-03    2.55211e-03
          10      2   1.03954e-02    5.07787e-03
          10      3   3.27767e-03    1.49876e-03
          30      1   9.98550e-03    3.31250e-03
          30      2   5.83415e-03    3.41400e-04
          30      3   2.63324e-03    2.08278e-04
          50      1   7.27525e-03    3.17039e-03
          50      2   1.10830e-02    3.18087e-03
          50      3   3.36324e-03    2.03456e-03

Energy Errors:
 Hidden Size  State Mean Energy Error Std. Dev. (Energy)
          10      1       1.79517e-03        1.09969e-03
          10      2       1.43003e-03        5.77397e-04
          10      3       2.31139e-03        1.13739e-03
          30      1       2.06465e-03        2.26875e-03
          30      2       2.36976e-03        2.04864e-03
          30      3       2.88010e-03        1.30081e-03
          50      1       1.34319e-03        1.13115e-03
          50      2       7

# Quantum Neural Network

In [9]:
# Set the number of qubits to 3
n_qubits = 3
# Create a quantum device with 3 qubits
dev1 = qml.device("default.qubit", wires=n_qubits)

def entangling_layer(nqubits):
    """
    Apply an entangling layer of CNOT gates to adjacent qubits.

    Args:
        nqubits (int): Number of qubits in use.
    """
    for i in range(0, nqubits - 1):
        qml.CNOT(wires=[i, i + 1])

def RY_layer(w):
    """
    Apply a layer of single-qubit rotations around the Y-axis (RY gates).

    Args:
        w (torch.Tensor): List of rotation angles for each qubit.
    """
    for idx, element in enumerate(w):
        qml.RY(element, wires=idx)

@qml.qnode(dev1)
def quantum_net(input_features, n_qubits, ansatz_depth, feature_weights, ansatz_weights, out_weights, lbd, Eweights):
    """
    Quantum circuit for computing wave function and energy level

    Args:
        input_features (torch.Tensor): Input features.
        n_qubits (int): Number of qubits in the circuit.
        ansatz_depth (int): Depth of the ansatz (number of layers).
        feature_weights (torch.Tensor): List of parameters.
        ansatz_weights (torch.Tensor): List of parameters.
        out_weights (torch.Tensor): List of parameters.
        lbd (torch.Tensor): A constant for computing energy.
        Eweights (torch.Tensor): parameter.
    Returns:
        tuple: Tuple of expectation values for Pauli-Z operators on qubits 0, 1, and 2 as wave function and energy level.
    """
    # apply feature map layers

    temp1 = Eweights[0]*lbd[0]
    qml.RY(temp1, wires=2)

    temp2 = feature_weights[0]*input_features[0]
    temp3 = feature_weights[1]*input_features[1]
    qml.RY(temp2, wires=0)
    qml.RY(temp3, wires=1)
    qml.CNOT(wires=[2, 0])
    qml.CNOT(wires=[2, 1])

    # apply ansatz layers
    for k in range(ansatz_depth):
        RY_layer(ansatz_weights[k]) # Apply RY gates
        qml.CNOT(wires=[2, 0])
        qml.CNOT(wires=[2, 1])
        
    # Compute expectation values for Pauli-Z operators
    exp_vals = [qml.expval(out_weights[0]*(qml.PauliZ(0)@qml.PauliZ(1))), qml.expval(Eweights[1]*qml.PauliZ(2))]
    return tuple(exp_vals)


class QNeuralNet(nn.Module):
    """
    quantum neural network class.

    Args:
        n_qubits (int): Number of qubits in the circuit.
        feature_depth (int): Depth of the feature map (number of layers).
        ansatz_depth (int): Depth of the ansatz (number of layers).
        domain (list): Defined domain in the interval [0, 2] for x and y axis.

    Notes:
        - The quantum neural network architecture consists of a single qubit for energy and two qubits for wave function.
        - The feature map has 'feature_depth' layers.
        - The ansatz has 'ansatz_depgh' layers.
    """
    def __init__(self, n_qubits, feature_depth, ansatz_depth, domain, seed):
        super(QNeuralNet, self).__init__()

        # Set the number qubits, number of ansatz layer and feature map layers
        self.n_qubits = n_qubits
        self.ansatz_depth = ansatz_depth
        self.feature_depth = feature_depth
        self.seed = seed
        
        # Initialize parameters of quantum circuit
        self.feature_weights = nn.Parameter(torch.FloatTensor(n_qubits-1, feature_depth).uniform_(3, 3.5))
        self.ansatz_weights = nn.Parameter(torch.randn(ansatz_depth*(n_qubits-1)).view(ansatz_depth, (n_qubits-1)))
        self.out_weights = nn.Parameter(torch.randn((1)).view(-1, 1))
        self.Energy = nn.Parameter(torch.tensor([3.14, 1.0]))

        # Set the x-borders and y-borders based on the specified domain
        self.xborders = domain[0]
        self.yborders = domain[1]

    def forward(self, input):
        """
        Forward pass through the neural network.

        Args:
            data (torch.Tensor): Input data.

        Returns:
            torch.Tensor: Outputs predictions from the neural network.
        """
        # Extract input features
        x = input[:, 0].view(-1, 1)
        y = input[:, 1].view(-1, 1)
        
        # Initialize an empty tensor for quantum output
        q_out = torch.Tensor(0, 2)
        # constant for computing energy
        lbd = torch.tensor([[1.0]])

        for elem in input:
            # Apply the quantum circuit (quantum_net) to each input element
            q_out_elem = torch.hstack(tuple(quantum_net(elem, self.n_qubits, self.ansatz_depth, self.feature_weights, self.ansatz_weights, self.out_weights, lbd, self.Energy))).float().unsqueeze(0)
            q_out = torch.cat((q_out, q_out_elem)) # Concatenate quantum outputs

        # Compute the ansatz solution by applying the transformation function to the neural network outputs for the wave function
        out = q_out[:, 0].view(-1, 1)
        energy = q_out[:, 1].view(-1, 1)
        return (1-torch.exp(-x+self.xborders[0]))*(1-torch.exp((x-self.xborders[1])))*(1-torch.exp(-y+self.yborders[0]))*(1-torch.exp((y-self.yborders[1])))*out, torch.abs(torch.mean(energy))

    def set_net_params(self):
        """
        set parameters in circuit.
        Args:
            None.
        """
        torch.manual_seed(self.seed)
        self.feature_weights.data = nn.Parameter(torch.FloatTensor(self.n_qubits-1, self.feature_depth).uniform_(3, 3.5))
        self.ansatz_weights.data = torch.randn(self.ansatz_depth * (self.n_qubits - 1)).view(self.ansatz_depth, (self.n_qubits - 1))
        self.out_weights.data = torch.randn(1).view(-1, 1)

    def set_energy_params(self, shooting):
        """
        set shooting number on energy qubit.

        Args:
            shooting (float): The value of shooting energy.
        """
        self.Energy = nn.Parameter(torch.tensor([3.14, shooting]))

In [10]:
ansatz_layers = [1, 2, 3]
quantum_model_list_seeds = []
quantum_history_lists_seeds = []
for seed in seeds:
    quantum_model_list = []
    quantum_history_lists = []
    for idx1, layer in enumerate(ansatz_layers):
        idx1 = idx1 + 1
        feature_depth, ansatz_depth = 1, layer
        energy_model_list = []
        for idx2 in range(energy_level):
            model_quantum = QNeuralNet(n_qubits, feature_depth, ansatz_depth, domain, seed)
            model_quantum.load_state_dict(torch.load(f"..\\trained_models\\quantum time-tndependent schrödinger\\{seed}\\quantum model_{seed}_{layer}{idx2}.pth"))
            energy_model_list.append(model_quantum)
        quantum_model_list.append(energy_model_list)
        quantum_history_lists.append(torch.load(f"..\\trained_models\\quantum time-tndependent schrödinger\\{seed}\\loss_Quantum model_{seed}_{layer}.pt"))        
    
    quantum_model_list_seeds.append(quantum_model_list)
    quantum_history_lists_seeds.append(quantum_history_lists)

In [11]:
results = []
for idx1 in range(len(quantum_model_list)):
    psi2_merror_list2 = []
    energy_error_list2 = []
    for idx2 in range(energy_level):
        psi2_merror_list3 = []
        energy_error_list3 = [] 
        for idx3, seed_name in enumerate(seeds):
            energy_error, psi2_merror = test_PINN(quantum_model_list_seeds[idx3][idx1][idx2], domain, num_test, E_exact_list[idx2], quantum_history_lists_seeds[idx3][idx1][idx2][-1], psi_exact_list[idx2])
            if energy_error >= 0.7:
                print("seed:",seed_name, "energy:", idx2+1, "Configuration:", idx1+1)
            else:
                psi2_merror_list3.append(psi2_merror)
                energy_error_list3.append(energy_error)
            
            
        mean_psi2 = np.mean(psi2_merror_list3)
        std_psi2 = np.std(psi2_merror_list3)
        mean_E = np.mean(energy_error_list3)
        std_E = np.std(energy_error_list3)
        
        psi2_merror_list2.append([idx2, mean_psi2, std_psi2])
        energy_error_list2.append([idx2, mean_E, std_E])

    results.append([idx1, psi2_merror_list2, energy_error_list2])

seed: 195 energy: 1 Configuration: 3
seed: 14 energy: 2 Configuration: 3
seed: 42 energy: 2 Configuration: 3
seed: 195 energy: 2 Configuration: 3
seed: 14 energy: 3 Configuration: 3
seed: 42 energy: 3 Configuration: 3
seed: 195 energy: 3 Configuration: 3


In [12]:
# Initialize lists to hold data for the two DataFrames
psi2_data = []
energy_data = []
# Loop over your results to extract and flatten the values
for idx1, psi2_list, energy_list in results:
    ansatz_layer = ansatz_layers[idx1]  # convert 0,1,2 → 10,30,50
    for state_idx, mean_psi2, std_psi2 in psi2_list:
        psi2_data.append([ansatz_layer, state_idx + 1, mean_psi2, std_psi2])
    for state_idx, mean_E, std_E in energy_list:
        energy_data.append([ansatz_layer, state_idx + 1, mean_E, std_E])

# Create DataFrames
df_psi2 = pd.DataFrame(psi2_data, columns=["Ansatz Depth", "State", "Mean ψ² Error", "Std. Dev. (ψ²)"])
df_energy = pd.DataFrame(energy_data, columns=["Ansatz Depth", "State", "Mean Energy Error", "Std. Dev. (Energy)"])

# Optional: Format with scientific notation
df_psi2["Mean ψ² Error"] = df_psi2["Mean ψ² Error"].map(lambda x: f"{x:.5e}")
df_psi2["Std. Dev. (ψ²)"] = df_psi2["Std. Dev. (ψ²)"].map(lambda x: f"{x:.5e}")
df_energy["Mean Energy Error"] = df_energy["Mean Energy Error"].map(lambda x: f"{x:.5e}")
df_energy["Std. Dev. (Energy)"] = df_energy["Std. Dev. (Energy)"].map(lambda x: f"{x:.5e}")

# Display the tables
print("ψ² Errors:")
print(df_psi2.to_string(index=False))

print("\nEnergy Errors:")
print(df_energy.to_string(index=False))

ψ² Errors:
 Ansatz Depth  State Mean ψ² Error Std. Dev. (ψ²)
            1      1   2.54618e-03    1.28147e-04
            1      2   6.11683e-03    1.95206e-04
            1      3   8.45717e-03    3.85510e-04
            2      1   2.81517e-03    3.42856e-04
            2      2   6.27514e-03    2.56887e-04
            2      3   8.04609e-03    2.36940e-05
            3      1   3.07697e-03    5.69708e-04
            3      2   6.12647e-03    0.00000e+00
            3      3   8.10107e-03    0.00000e+00

Energy Errors:
 Ansatz Depth  State Mean Energy Error Std. Dev. (Energy)
            1      1       1.54448e-03        6.96046e-04
            1      2       5.08547e-03        1.61681e-03
            1      3       7.52378e-03        1.41142e-03
            2      1       1.53852e-03        4.24760e-04
            2      2       3.69883e-03        1.43611e-03
            2      3       7.75456e-03        8.65275e-04
            3      1       2.90513e-03        8.16482e-04
         

# Hybrid Quantum Neural Network

In [13]:
# Set the number of qubits to 3
n_qubits = 3
# Create a quantum device with 3 qubits
dev = qml.device("default.qubit", wires=n_qubits)

def RY_layer(w):
    """
    Apply a layer of single-qubit rotations around the Y-axis (RY gates).

    Args:
        w (torch.Tensor): List of rotation angles for each qubit.
    """
    for idx, element in enumerate(w):
        qml.RY(element, wires=idx)

def entangling_layer(nqubits):
    """
    Apply an entangling layer of CNOT gates to adjacent qubits.

    Args:
        nqubits (int): Number of qubits in use.
    """
    for i in range(0, nqubits - 1):
        qml.CNOT(wires=[i, i + 1])

@qml.qnode(dev)
def hquantum_net(input_features, n_qubits, q_depth, q_weights):
    """
    Quantum circuit as a part of quantum hybrid neural network.

    Args:
        input_features (torch.Tensor): Input features.
        n_qubits (int): Number of qubits in the circuit.
        q_depth (int): Number of quantum layers.
        q_weights (torch.Tensor): List of parameters.
    Returns:
        tuple: Tuple of expectation values for Pauli-Z operators on qubits 0, 1, and 2.
    """
    RY_layer(input_features[0])
    RY_layer(q_weights[0])

    entangling_layer(n_qubits)

    RY_layer(input_features[1])
    RY_layer(q_weights[1])

    exp_vals = [qml.expval(qml.PauliZ(position)) for position in range(0, n_qubits)]
    return tuple(exp_vals)

class HQNeuralNet(nn.Module):
    """
    quantum hybrid neural network class.

    Args:
        input_size (int): Size of the input features.
        hidden_size (int): Number of hidden units in the neural network.
        n_qubits (int): Number of qubits in the circuit.
        q_depth (int): Number of quantum layers.
        output_size (int): Size of the output.
        domain (list): Defined domain in the interval [0, 2] for x and y axis.

    Attributes:
        fnn (nn.Sequential): Feedforward neural network with a linear layer followed by a Tanh activation.

    Notes:
        - The first classical neural network architecture consists of a one linear layer for energy and a single hidden layers for wave function.
        - The input layer has 'input_size' neurons.
        - The hidden layers for predicting wave function has 'hidden_size' neurons with a Tanh activation.
        - The quantum neural network architecture consists of 'n_qubits' and 'q_depth ' layers.
        - The second classical neural network architecture consists of one linear layer.
        - The output layer has 'output_size' neurons.
    """
    def __init__(self, input_size, hidden_size, n_qubits, q_depth, output_size, domain, seed):
        super(HQNeuralNet, self).__init__()
        # Define the feedforward (1, 1) layer
        self.Energy = nn.Linear(1, 1)
        
        # Define the feedforward neural network (fnn)
        self.fnn = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, (q_depth)*n_qubits),
        )
        
        #Define the feedforward (n_qubits, output_size) layer as a output layer
        self.output_layer = nn.Linear(n_qubits, output_size)

        # Set the number qubits, number of quantum layers
        self.seed = seed
        self.n_qubits = n_qubits
        self.q_depth = q_depth
        
        # Initialize parameters of quantum circuit
        self.q_weights = nn.Parameter(0.2*torch.randn(q_depth * n_qubits).view(q_depth, n_qubits))

        # Set the x-borders and y-borders based on the specified domain
        self.xborders = domain[0]
        self.yborders = domain[1]

    def forward(self, data):
        """
        Forward pass through the neural network.

        Args:
            data (torch.Tensor): Input data.

        Returns:
            torch.Tensor: Outputs predictions from the neural network.
        """
        # Extract input features
        x, y= data[:, 0].view(-1, 1), data[:, 1].view(-1, 1)
        
        # set lambda to pass through the linear layer (energy layer)
        lbd = torch.tensor([[1.0]])
        Energy = self.Energy(lbd)
        
        # Pass the data through the feedforward neural network (self.fnn)
        out = self.fnn(data)
        # Initialize an empty tensor for quantum output
        q_out = torch.Tensor(0, self.n_qubits)
        
        for elem in out:
            # Apply the quantum circuit (hquantum_net) to each output element
            elem = elem.view((self.q_depth), self.n_qubits)
            q_out_elem = torch.hstack(tuple(hquantum_net(elem, self.n_qubits, self.q_depth, self.q_weights))).float().unsqueeze(0)
            q_out = torch.cat((q_out, q_out_elem))# Concatenate quantum outputs

        # Pass the output of quantum circuit through the feedforward linear layer then use ansatz solution
        out = self.output_layer(q_out)
        out1 = out[:, 0].view(-1, 1)
        out2 = out[:, 1].view(-1, 1)
        out = (1-torch.exp(-x+self.xborders[0]))*(1-torch.exp((x-self.xborders[1])))*out1*(1-torch.exp(-y+self.yborders[0]))*(1-torch.exp((y-self.yborders[1])))*out2
        return out, torch.abs(Energy)

    def init_weights(self, m):
        """
        Initializes weights and biases for a linear layer.

        Args:
            m (nn.Linear): The linear layer to initialize.

        Notes:
            - Applies uniform weight initialization within the range [-n0, n0].
            - Initializes biases with a constant value of 0.01.
        """
        if isinstance(m, nn.Linear):
            in_sz = torch.tensor(m.in_features)
            out_sz = torch.tensor(m.out_features)
            n0 = torch.sqrt(6/(in_sz+out_sz))
            m.weight.data.uniform_(-n0, n0)
            m.bias.data.fill_(0.01)

    def init_energy(self, shooting):
        """
        Initializes weights and biases for a linear layer.

        Args:
            m (nn.Linear): The linear layer to initialize.

        Notes:
            - Applies shooting number.
            - Initializes biases with a constant value of 0.0.
        """
        def inner(m):
            if isinstance(m, nn.Linear):
                m.weight.data.fill_(shooting)
                m.bias.data.fill_(0.0)
        return inner

    def set_net_params(self):
        """
        set weights and biases for linear layers and parameters in the circuit.

        Args:
            None.
        """
        # Set the random seed for reproducibility
        torch.manual_seed(self.seed)
        
        self.fnn.apply(self.init_weights)
        self.q_weights = nn.Parameter(0.2*torch.randn(q_depth * n_qubits).view(q_depth, n_qubits))
        self.output_layer.apply(self.init_weights)

    def set_energy_params(self, shooting):
        """
        set shooting number in energy qubit.

        Args:
            shooting (float): The value of shooting energy.
        """
        self.Energy.apply(self.init_energy(shooting))
        

In [14]:
hybrid_model_list_seeds = []
hybrid_history_lists_seeds = []
for seed in seeds:       
    hybrid_model_list = []
    hybrid_history_lists = []
    for idx1, hs in enumerate([10, 30, 50]):
        idx1 = idx1 + 1
        input_size, hidden_size, q_depth, output_size = 2, hs, 2, 2
        energy_model_list = []
        for idx2 in range(energy_level):
            model_hybrid = HQNeuralNet(input_size, hidden_size, n_qubits, q_depth, output_size, domain, seed)
            model_hybrid.load_state_dict(torch.load(f"..\\trained_models\\hybrid time-tndependent schrödinger\\{seed}\\\\Hybrid model_{seed}_{hs}{idx2}.pth"))
            energy_model_list.append(model_hybrid)
        hybrid_model_list.append(energy_model_list)
        hybrid_history_lists.append(torch.load(f"..\\trained_models\\hybrid time-tndependent schrödinger\\{seed}\\\\loss_Hybrid model_{seed}_{hs}.pt"))    


    hybrid_model_list_seeds.append(hybrid_model_list)
    hybrid_history_lists_seeds.append(hybrid_history_lists)

In [15]:
results = []
for idx1 in range(len(hybrid_model_list)):
    psi2_merror_list2 = []
    energy_error_list2 = []
    for idx2 in range(energy_level):
        psi2_merror_list3 = []
        energy_error_list3 = [] 
        for idx3, seed_name in enumerate(seeds):
            energy_error, psi2_merror = test_PINN(hybrid_model_list_seeds[idx3][idx1][idx2], domain, num_test, E_exact_list[idx2], hybrid_history_lists_seeds[idx3][idx1][idx2][-1], psi_exact_list[idx2])
            if energy_error >= 0.7:
                print("seed:",seed_name, "energy:", idx2+1, "Configuration:", idx1+1)
            else:
                psi2_merror_list3.append(psi2_merror)
                energy_error_list3.append(energy_error)
            
        mean_psi2 = np.mean(psi2_merror_list3)
        std_psi2 = np.std(psi2_merror_list3)
        mean_E = np.mean(energy_error_list3)
        std_E = np.std(energy_error_list3)
        
        psi2_merror_list2.append([idx2, mean_psi2, std_psi2])
        energy_error_list2.append([idx2, mean_E, std_E])

    results.append([idx1, psi2_merror_list2, energy_error_list2])

seed: 42 energy: 3 Configuration: 1
seed: 86 energy: 2 Configuration: 3
seed: 14 energy: 3 Configuration: 3
seed: 42 energy: 3 Configuration: 3


In [16]:
# Initialize lists to hold data for the two DataFrames
psi2_data = []
energy_data = []

# Loop over your results to extract and flatten the values
for idx1, psi2_list, energy_list in results:
    hidden_size = hidden_sizes[idx1]  # convert 0,1,2 → 10,30,50
    for state_idx, mean_psi2, std_psi2 in psi2_list:
        psi2_data.append([hidden_size, state_idx + 1, mean_psi2, std_psi2])
    for state_idx, mean_E, std_E in energy_list:
        energy_data.append([hidden_size, state_idx + 1, mean_E, std_E])

# Create DataFrames
df_psi2 = pd.DataFrame(psi2_data, columns=["Hidden Size", "State", "Mean ψ² Error", "Std. Dev. (ψ²)"])
df_energy = pd.DataFrame(energy_data, columns=["Hidden Size", "State", "Mean Energy Error", "Std. Dev. (Energy)"])

# Optional: Format with scientific notation
df_psi2["Mean ψ² Error"] = df_psi2["Mean ψ² Error"].map(lambda x: f"{x:.5e}")
df_psi2["Std. Dev. (ψ²)"] = df_psi2["Std. Dev. (ψ²)"].map(lambda x: f"{x:.5e}")
df_energy["Mean Energy Error"] = df_energy["Mean Energy Error"].map(lambda x: f"{x:.5e}")
df_energy["Std. Dev. (Energy)"] = df_energy["Std. Dev. (Energy)"].map(lambda x: f"{x:.5e}")

# Display the tables
print("ψ² Errors:")
print(df_psi2.to_string(index=False))

print("\nEnergy Errors:")
print(df_energy.to_string(index=False))

ψ² Errors:
 Hidden Size  State Mean ψ² Error Std. Dev. (ψ²)
          10      1   1.82040e-03    2.50753e-04
          10      2   7.69070e-03    2.71132e-03
          10      3   2.50288e-03    9.12585e-04
          30      1   1.70200e-03    2.88978e-04
          30      2   6.86472e-03    1.26645e-03
          30      3   1.91672e-03    3.63994e-04
          50      1   1.50241e-03    4.38189e-04
          50      2   4.33279e-03    1.72026e-03
          50      3   2.79699e-03    1.31317e-04

Energy Errors:
 Hidden Size  State Mean Energy Error Std. Dev. (Energy)
          10      1       4.97222e-04        2.65763e-04
          10      2       1.72377e-04        1.19406e-04
          10      3       1.11230e-03        1.15176e-03
          30      1       4.91858e-04        4.77363e-04
          30      2       3.96490e-04        1.22020e-04
          30      3       1.03402e-03        3.47331e-04
          50      1       2.99394e-04        2.85087e-04
          50      2       2